# 16장 · Function Calling의 실행과 결과 반환

도구 실행을 맡은 프로그램은 인자를 검사하고 함수를 호출한 뒤 결과에 원래 호출 ID를 붙입니다. 모델은 이 ID로 요청과 결과를 짝짓습니다.

15장의 호출 요청을 검증한 뒤 도서관 시간표 함수를 실행합니다. 요청과 결과의 call_id를 대조하고, 허용 범위를 벗어난 요일의 처리 결과를 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 도구 이름과 인자를 함수 호출로 변환

명세만 본 모델은 운영 시간을 알 수 없습니다. 읽기 전용 조회 함수 안의 표에서 시간을 꺼내면 비로소 답에 쓸 값이 생깁니다.

### 셀 01 · get_library_hours()의 반환값

이번 셀에서는 모델 없이 파이썬 함수를 직접 부릅니다.

In [1]:
def get_library_hours(day):
    hours = {"weekday": "09:00~18:00", "weekend": "10:00~16:00"}
    return {"day": day, "hours": hours[day], "source": "수업용 가상 도서관"}

print(get_library_hours("weekend")["hours"])

10:00~16:00


10:00~16:00은 주말 표에서 꺼낸 값입니다. 이 함수는 실제 도서관에 접속하지 않습니다. 허용되지 않은 인자가 들어오는지는 함수를 부르기 전에 검사합니다.

같은 weekend 조회를 두 번 보내면 두 함수 결과의 시간은 같을 수 있습니다. 이때 10:00~16:00만으로는 어느 요청의 답인지 가릴 수 없습니다. 호출자가 보낸 call_id를 결과에 함께 돌려줘야 각각의 요청과 짝을 지을 수 있습니다.

### 읽기 전용 함수 연결

<p>15장의 요일 허용 목록을 이번에는 execute_call에서 검사한 뒤 시간표를 조회합니다.</p>

### 셀 02 · execute_call()과 결과 직렬화

제공 함수는 요청의 이름과 인자를 검사합니다. 성공했을 때 어떤 값이 output 안에 담기는지 읽으세요.

이번 노트북은 주말 조회 요청을 새로 만들어 call_1이라는 ID를 붙입니다. 15장의 예제 ID는 call_demo_1이었습니다. 이제부터 이 요청에 답하는 결과는 call_1을 그대로 사용해야 합니다.

In [2]:
import json

def execute_call(item):
    if not isinstance(item.get("call_id"), str) or not item["call_id"].strip():
        raise ValueError("call_id 없음 또는 공백: 결과 연결 불가")
    try:
        if item.get("name") != "get_library_hours":
            raise ValueError("허용하지 않은 도구")
        args = json.loads(item["arguments"])
        if not isinstance(args, dict) or set(args) != {"day"}:
            raise ValueError("인자 필드 불일치")
        if args["day"] not in {"weekday", "weekend"}:
            raise ValueError("지원하지 않는 요일 값")
        result = {"ok": True, "data": get_library_hours(args["day"])}
    except (ValueError, TypeError, KeyError):
        result = {"ok": False, "error": "invalid_tool_request"}
    return {"type": "function_call_output", "call_id": item["call_id"],
            "output": json.dumps(result, ensure_ascii=False)}

call = {"type": "function_call", "call_id": "call_1", "name": "get_library_hours", "arguments": '{"day":"weekend"}'}
reply = execute_call(call)
print(json.loads(reply["output"])["data"]["hours"])

10:00~16:00


같은 10:00~16:00이 나오지만 이번 값은 reply의 output에 담긴 JSON 문자열에서 읽었습니다. 모델에 보낼 항목의 type은 function_call_output입니다.

#### 같은 시간표 요청이 두 개라면

<mark>도구 결과를 원래 요청과 잇는 값은 call_id입니다.</mark> 두 응답이 늦게 또는 다른 순서로 도착할 수 있습니다. 함수 이름이 같아도 각각의 호출 ID를 보관해야 어느 질문에 대한 운영 시간인지 복원할 수 있습니다.

선수 개념 · 15장 Function Calling의 도구 명세와 호출 요청

#### 허용하지 않은 요일에도 요청 ID가 남는 이유

```mermaid
flowchart TD
ncall["요청"]
  nvalidate["day 검사"]
  nlookup["시간 조회"]
  nerror["오류 구성"]
  noutput["결과 연결"]
  ncall --> nvalidate
  nvalidate --> nlookup
  nvalidate --> nerror
  nlookup --> noutput
  nerror --> noutput
```

day가 Friday인 요청은 조회 함수를 실행하기 전에 걸러집니다. 결과의 ok는 False이지만 바깥 call_id에는 원래 요청 ID가 남습니다.

### ID와 인자를 거부하는 방법의 차이

<p>execute_call은 호출 ID가 비어 있으면 ValueError를 일으킵니다. ID가 있어 결과를 연결할 수 있지만 이름이나 day가 잘못됐다면, 호출 ID가 포함된 function_call_output에 invalid_tool_request를 담아 반환합니다.</p>

## function_call_output과 call_id 대응

결과를 돌려주는 쪽은 임의의 새 ID를 붙이지 않습니다. 모델이 보낸 호출 항목의 call_id를 그대로 써야 요청과 결과가 한 쌍이 됩니다.

### 허용하지 않은 요일에도 요청 ID가 남는 이유

day가 Friday인 요청은 조회 함수를 실행하기 전에 걸러집니다. 결과의 ok는 False이지만 바깥 call_id에는 원래 요청 ID가 남습니다.

1. 요청 · call_id = call_1
2. day 검사 · day = Friday
3. 오류 구성 · error = invalid_tool_request
4. 결과 연결 · call_id = call_1

### 과제 03 · call_id 일치 여부

원본 call과 결과 reply의 ID가 같은지 확인하고, 같으면 그 ID를 출력하세요.

In [3]:
# 두 ID가 같은지 검사하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>assert reply[&quot;call_id&quot;] == call[&quot;call_id&quot;]
print(reply[&quot;call_id&quot;])</code></pre>
<p>실행 결과</p><pre><code>call_1</code></pre>
<p>call_1입니다. 모델에게 다음 답변을 요청할 때는 사용자 질문과 호출 항목 다음에 이 결과를 넣습니다.</p>
</details>

### function_call_output의 두 층

<p>reply의 call_id는 바깥 딕셔너리에서 바로 읽습니다. 반면 output에는 성공 여부와 조회 데이터가 JSON 문자열로 들어 있습니다. json.loads(reply['output'])을 거치면 data에서 hours를 꺼낼 수 있습니다.</p>

### 셀 04 · 허용 목록 밖 인자 거부

이번 요청의 ID는 유효하나 day가 명세의 두 선택지 밖에 있습니다.

In [4]:
bad = {**call, "arguments": '{"day":"Friday"}'}
rejected = execute_call(bad)
print(json.loads(rejected["output"]))

{'ok': False, 'error': 'invalid_tool_request'}


ok가 False이고 error가 invalid_tool_request입니다. 허용되지 않은 요일로 얻은 시간을 사용자에게 보여 주는 일을 막았습니다.

### 오류 결과의 전달

<table><caption>실패 사례</caption><thead><tr><th scope="col">입력</th><th scope="col">예상 처리</th></tr>
</thead><tbody><tr><td>등록하지 않은 함수 이름</td><td>오류 결과, 함수 실행 없음</td></tr>
<tr><td>day=holiday</td><td>오류 결과</td></tr>
<tr><td>JSON으로 읽히지 않는 인자</td><td>오류 결과</td></tr>
<tr><td>불필요한 extra 필드</td><td>오류 결과</td></tr>
<tr><td>call_id 누락·공백·문자열 아님</td><td>연결 불가 예외, 도구 실행 없음</td></tr>

</tbody></table>

## 대화 기록에 호출과 결과 추가

도구 호출을 제안한 응답과 함수 실행 결과를 대화 끝에 차례로 넣습니다. 모델에 다시 보내기 직전에는 결과가 어느 항목인지, 조회 실패를 개관 시간으로 오해하지 않았는지 살펴보세요.

### 질문 뒤에 호출 항목과 결과 붙이기

<p>history에는 사용자 질문, 모델의 호출 요청, 프로그램이 만든 결과가 차례대로 들어갑니다. 마지막 항목에서 function_call_output을 읽고 같은 call_id의 이전 요청을 찾아 연결해 보세요.</p>

### 셀 05 · append()로 호출 이력 구성

대화 내역을 만든 뒤 마지막 항목의 유형을 읽습니다.

In [5]:
history = [{"role": "user", "content": "주말 운영 시간은?"}]
history.append(call)
history.append(reply)
print(history[-1]["type"])

function_call_output


function_call_output이 나옵니다. 모델의 최종 메시지는 아직 없습니다. 다음 과제에서는 조회 값으로 안내 문장을 직접 만듭니다.

### 도구 왕복의 모델·프로그램 역할

<p>history의 마지막 항목은 프로그램이 만든 조회 결과입니다. 모델을 다시 호출하는 API 코드는 이 노트북에 없으므로, 과제 06에서는 그 항목을 읽어 안내 문장을 직접 조립합니다.</p>

<p>표의 시간은 실습용입니다. 실제로 실행한 부분은 입력 검사와 시간표 조회, JSON 직렬화입니다.</p>

### 과제 06 · 도구 출력에서 최종 답변 조립

현재 대화의 마지막 항목을 읽어 성공한 결과일 때만 시간을 넣어 안내하세요. 이 문장은 실제 모델 답변 대신 쓰는 연습용 문구입니다.

In [6]:
# history에서 성공 여부와 시간을 읽으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>payload = json.loads(history[-1][&quot;output&quot;])
if payload[&quot;ok&quot;]:
    print(&quot;가상 주말 운영 시간:&quot;, payload[&quot;data&quot;][&quot;hours&quot;])
else:
    print(&quot;운영 시간을 확인하지 못했습니다.&quot;)</code></pre>
<p>실행 결과</p><pre><code>가상 주말 운영 시간: 10:00~16:00</code></pre>
<p>예제에서는 ‘가상 주말 운영 시간: 10:00~16:00’이 나옵니다. 한 번의 왕복에서 도구 실행은 한 번이고, 실제 SDK를 쓰는 구성이라면 최초 요청과 최종 답 요청을 따로 셉니다.</p>
</details>

## 돌아보기

호출 ID가 빠지면 어느 요청의 결과인지 정할 수 없습니다. 두 요청이 동시에 대기 중인 아래 예제로 결과의 도착 순서와 ID를 대조해 보세요.

### 과제 07 · 응답 순서가 뒤집힌 두 요청

주말 요청 뒤에 평일 요청을 보냈는데 평일 결과가 먼저 도착했습니다. 아래 셀에서 두 결과를 call_id로 찾아 요청 순서대로 운영 시간을 출력하세요.

In [ ]:
weekday = {**call, "call_id": "call_2", "arguments": '{"day":"weekday"}'}
arrivals = [execute_call(weekday), reply]
# 요청 순서(call, weekday)대로 두 결과를 찾아 출력하세요.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>weekday = {**call, "call_id": "call_2", "arguments": '{"day":"weekday"}'}
arrivals = [execute_call(weekday), reply]

by_call = {result["call_id"]: result for result in arrivals}
for request in (call, weekday):
    payload = json.loads(by_call[request["call_id"]]["output"])
    print(request["call_id"], payload["data"]["hours"])</code></pre>
<p>실행 결과</p><pre><code>call_1 10:00~16:00
call_2 09:00~18:00</code></pre>
<p>먼저 도착한 결과의 ID는 call_2입니다. 위치 0을 첫 요청에 붙이면 주말 질문에 평일 시간을 답하게 됩니다.</p>
</details>

## 대출 권수 조회를 기존 시간표 도구 옆에 연결하기

주말 운영 시간과 한 번에 빌릴 수 있는 책의 수는 다른 자료입니다. 기존 execute_call에 새 이름을 보내 어디서 멈추는지 확인한 뒤, 대출 권수 요청을 처리할 함수를 연결하세요. 권수는 수업용 가상 규칙입니다.

### 셀 09 · 새 요청을 기존 실행기로 보내기

대출 권수 도구는 추가 인자를 받지 않습니다. 기존 실행기에 이 이름을 보내고 호출 ID와 결과를 읽으세요.

In [7]:
limit_call = {"type": "function_call", "name": "get_loan_limit",
              "call_id": "loan-3", "arguments": "{}"}
old_result = execute_call(limit_call)
print(old_result["call_id"], json.loads(old_result["output"]))

loan-3 {'ok': False, 'error': 'invalid_tool_request'}


loan-3이라는 ID를 돌려받았지만 ok는 False입니다. 호출이 도착한 것과 등록한 함수가 성공한 것은 다릅니다.

### 셀 10 · 권수 조회에 필요한 검사와 반환값 준비하기

새 실행기가 연결할 함수입니다. 호출 ID가 있어야 하고 인자는 빈 JSON 객체여야 합니다. 함수가 만든 데이터에서 한 번에 빌릴 수 있는 권수를 읽으세요.

In [8]:
def loan_limit_reply(item):
    call_id = item.get("call_id")
    if not isinstance(call_id, str) or not call_id.strip():
        raise ValueError("호출 ID가 필요합니다.")
    try:
        if json.loads(item["arguments"]) != {}:
            raise ValueError("권수 조회에는 인자가 필요 없습니다.")
        result = {"ok": True, "data": {"max_books": 3, "source": "가상 대출 규칙"}}
    except (ValueError, KeyError, TypeError):
        result = {"ok": False, "error": "invalid_tool_request"}
    return {"type": "function_call_output", "call_id": call_id,
            "output": json.dumps(result, ensure_ascii=False)}

print(json.loads(loan_limit_reply(limit_call)["output"])["data"]["max_books"])

3


3은 가상 대출 규칙에서 정한 권수입니다. 이 함수는 자신에게 이미 적절한 이름의 호출이 전달됐다고 가정합니다. 이름으로 함수를 고르는 일은 다음 과제에서 맡습니다.

### 과제 11 · 호출 이름으로 두 실행기 연결하기

새 이름은 loan_limit_reply에 보내고 나머지는 기존 execute_call에 맡기는 execute_library를 작성하세요. 대출 권수와 기존 주말 시간 요청을 실행해 각각의 요청 ID와 조회값을 출력하세요.

In [9]:
def execute_library(item):
    pass  # 요청의 name을 읽고 연결할 함수를 선택하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def execute_library(item):
    if item.get(&quot;name&quot;) == &quot;get_loan_limit&quot;:
        return loan_limit_reply(item)
    return execute_call(item)

limit_reply = execute_library(limit_call)
hours_reply = execute_library(call)
print(limit_reply[&quot;call_id&quot;], json.loads(limit_reply[&quot;output&quot;])[&quot;data&quot;][&quot;max_books&quot;])
print(hours_reply[&quot;call_id&quot;], json.loads(hours_reply[&quot;output&quot;])[&quot;data&quot;][&quot;hours&quot;])</code></pre>
<p>실행 결과</p><pre><code>loan-3 3
call_1 10:00~16:00</code></pre>
<p>loan-3 요청은 3권을, call_1 요청은 주말 시간표를 돌려줍니다. 기존 함수가 모르는 이름을 거절하던 경로도 유지됩니다. 주말 시간이 나오는 줄까지 확인해야 새 도구를 붙이면서 원래 동작을 잃지 않았는지 알 수 있습니다.</p>
</details>

<mark>도구 이름은 실행할 함수를 고르고, 호출 ID는 그 실행 결과를 원래 요청에 붙입니다.</mark> 권수 응답을 받아도 시간표를 물은 요청에 붙일 수는 없습니다. 두 도구가 성공했는지 살핀 뒤 각 결과 안에 남은 ID를 요청 쪽 ID와 맞춰 보세요.

모델이 한 번에 두 도구를 요청하면 같은 방식으로 각각을 실행해 두 결과를 한 대화 이력에 넣을 수 있습니다.

## 실제 모델에 도구 결과를 돌려주기

응답에 포함된 호출 ID를 사용해 함수 실행 결과를 다시 전달합니다. 첫 응답의 인자와 최종 안내 시간이 실제 함수 결과와 맞는지 확인하세요.

In [ ]:
import os
from openai import OpenAI
client = OpenAI(timeout=45, max_retries=1)
model_name = os.getenv("OPENAI_MODEL") or "gpt-4o-mini"
TOOL = {
    "type": "function", "name": "get_library_hours",
    "description": "가상 도서관의 요일별 운영 시간을 조회합니다. 실제 예약이나 계정 변경은 하지 않습니다.",
    "parameters": {
        "type": "object",
        "properties": {"day": {"type": "string", "enum": ["weekday", "weekend"]}},
        "required": ["day"], "additionalProperties": False,
    },
    "strict": True,
}
reply = client.responses.create(model=model_name, tools=[TOOL],
    instructions="도서관 운영 시간은 등록된 도구를 조회하세요.",
    input="주말에 몇 시에 문을 여나요?")
calls = [item for item in reply.output if item.type == "function_call"]
for item in calls:
    print(item.name, item.arguments, item.call_id)
if calls:
    results = [execute_call(item.model_dump()) for item in calls]
    final = client.responses.create(model=model_name, previous_response_id=reply.id, input=results)
    print(final.output_text)
else:
    print("도구 요청 없음", reply.output_text)

get_library_hours {"day":"weekend"} call_fvGla5dc4bAFuoTEJ8sQmTsS
주말에는 오전 10시부터 오후 4시까지 문을 엽니다.
